# RRN GPU training — staged Vimeo-90K LMDB shards
Run each cell in order. The smoke test copies one shard to `/content` and runs 20 T4 steps. It never trains directly from Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lmdb


In [ ]:
from pathlib import Path
import json, subprocess
PROJECT = Path('/content/drive/MyDrive/train_ai/projects/RNN')
ADAPTER_REPO = 'https://github.com/koorock92/rrn-colab-adapter.git'
ADAPTER = Path('/content/rrn-colab-adapter')
UPSTREAM = Path('/content/RRN-upstream')
UPSTREAM_COMMIT = 'db440c5d363dfe6c11bdd6c82e63781b295e6211'
if not ADAPTER.exists():
    subprocess.run(['git', 'clone', '--depth', '1', ADAPTER_REPO, str(ADAPTER)], check=True)
else:
    subprocess.run(['git', '-C', str(ADAPTER), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(ADAPTER), 'reset', '--hard', 'origin/main'], check=True)
if not UPSTREAM.exists():
    subprocess.run(['git', 'clone', 'https://github.com/junpan19/RRN.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', UPSTREAM_COMMIT], check=True)
WORK = ADAPTER / 'src'
SHARDS = Path('/content/drive/MyDrive/train_ai/datasets/vimeo90k/vimeo90k_shards')
CACHE = Path('/content/rrn_shard_cache')
SMOKE_OUT = PROJECT / 'logs/sharded_gpu_smoke'
FULL_OUT = PROJECT / 'logs/sharded_gpu_full'
manifest = json.loads((SHARDS / 'manifest.json').read_text())
print('train shards:', len(manifest['train_shards']))
print('train samples:', manifest['train_samples'])
print('validation samples:', manifest['validation_samples'])


In [ ]:
import torch
assert torch.cuda.is_available(), '請先選 執行階段 > 變更執行階段類型 > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!df -h / /content/drive
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv


## 20-step real-model smoke test
This uses 128 channels and 10 blocks. The input pipeline automatically selects torchvision's native PNG decoder (or optional pyspng) and prefetches four batches per worker. `--no-prefetch` only disables next-shard copying.

In [ ]:
import subprocess, sys
last = SMOKE_OUT / 'last.pt'
cmd = [sys.executable, str(WORK / 'train_rrn_sharded.py'),
       '--upstream-dir', str(UPSTREAM), '--shards-root', str(SHARDS), '--cache-dir', str(CACHE), '--output', str(SMOKE_OUT),
       '--device', 'cuda', '--scale', '4', '--crop-size', '64', '--batch-size', '4',
       '--workers', '2', '--channels', '128', '--blocks', '10', '--epochs', '70',
       '--save-every', '10', '--max-steps', '20', '--decoder', 'auto', '--prefetch-factor', '4',
       '--amp', '--no-prefetch']
if last.exists():
    cmd += ['--resume', str(last)]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
import json
lines = (SMOKE_OUT / 'metrics.jsonl').read_text().splitlines()
recent = [json.loads(line) for line in lines[-20:]]
print('steps:', len(recent))
print('mean seconds/step:', sum(x['seconds'] for x in recent) / len(recent))
print('last:', SMOKE_OUT / 'last.pt', (SMOKE_OUT / 'last.pt').exists())
print('best:', SMOKE_OUT / 'best.pt', (SMOKE_OUT / 'best.pt').exists())
!df -h / /content/drive
!nvidia-smi --query-gpu=name,memory.used,utilization.gpu --format=csv


## Full training (batch 16, AMP, prefetch enabled; leave disabled until confirmed)

In [ ]:
CONFIRM_FULL_TRAINING = False
if not CONFIRM_FULL_TRAINING:
    print('尚未啟動正式訓練。先確認上方 20-step 平均速度。')
else:
    import subprocess, sys, math
    from collections import deque
    from html import escape
    from IPython.display import HTML, display
    last = FULL_OUT / 'last.pt'
    cmd = [sys.executable, str(WORK / 'train_rrn_sharded.py'),
           '--upstream-dir', str(UPSTREAM), '--shards-root', str(SHARDS), '--cache-dir', str(CACHE), '--output', str(FULL_OUT),
           '--device', 'cuda', '--scale', '4', '--crop-size', '64', '--batch-size', '16',
           '--workers', '2', '--channels', '128', '--blocks', '10', '--epochs', '70',
           '--save-every', '1000', '--metrics-flush-every', '100', '--log-every', '25',
           '--decoder', 'auto', '--prefetch-factor', '4', '--gpu-downsample', '--amp']
    if last.exists():
        cmd += ['--resume', str(last)]
    print(' '.join(cmd), flush=True)
    ui = {'epoch': '-', 'shard': '-', 'step': 0, 'epoch_batch': 0, 'epoch_batches': 1,
          'loss': '--', 'mae': '--', 'lr': '--', 'psnr': '--', 'best': '--',
          'data_s': '--', 'compute_s': '--', 'wall_s': '--', 'samples_s': '--',
          'eta': '--', 'checkpoint': 0, 'decoder': 'auto',
          'stage': 'Shard cache ready', 'stage_percent': 100.0, 'event': 'Starting trainer...'}
    recent_wall = deque(maxlen=20)
    status = display(HTML('Starting...'), display_id=True)
    def fields(line):
        return dict(token.split('=', 1) for token in line.split()[1:] if '=' in token)
    def finite_metric(value, digits=4):
        try:
            number = float(value)
            return f'{number:.{digits}f}' if math.isfinite(number) else '--'
        except (TypeError, ValueError):
            return '--'
    def duration(seconds):
        seconds = max(0, int(seconds))
        hours, seconds = divmod(seconds, 3600)
        minutes, seconds = divmod(seconds, 60)
        return f'{hours:02d}:{minutes:02d}:{seconds:02d}'
    def render(color='#22c55e'):
        percent = 100.0 * ui['epoch_batch'] / max(ui['epoch_batches'], 1)
        stage = '' if ui['stage_percent'] >= 100 else f"""
        <div style='margin-top:10px;color:#fbbf24'>{escape(ui['stage'])}</div>
        <div style='height:6px;background:#374151;border-radius:4px;margin-top:4px'>
        <div style='height:6px;width:{ui['stage_percent']:.2f}%;background:#f59e0b;border-radius:4px'></div></div>"""
        status.update(HTML(f"""<div style='font-family:ui-monospace,SFMono-Regular,Consolas,monospace;padding:16px;background:#111827;color:#e5e7eb;border-radius:10px;line-height:1.55'>
        <div style='font-size:15px;font-weight:700;color:#f9fafb'>RRN ×4 · Epoch {escape(str(ui['epoch']))} · Shard {escape(str(ui['shard']))} · Step {ui['step']:,}</div>
        <div>PSNR: <b style='color:#60a5fa'>{ui['psnr']}</b> · Best: <b style='color:#a78bfa'>{ui['best']}</b> · MAE: {ui['mae']} · Loss: {ui['loss']} · LR: {ui['lr']}</div>
        <div style='height:14px;background:#374151;border-radius:7px;margin:9px 0 5px'>
        <div style='height:14px;width:{percent:.2f}%;background:{color};border-radius:7px'></div></div>
        <div>{ui['epoch_batch']:,}/{ui['epoch_batches']:,} ({percent:.1f}%) · {ui['wall_s']} s/step · {ui['samples_s']} clips/s · ETA {ui['eta']}</div>
        <div style='color:#9ca3af'>Data {ui['data_s']}s · GPU compute {ui['compute_s']}s · decoder {escape(ui['decoder'])} · checkpoint step {ui['checkpoint']:,}</div>
        {stage}<div style='margin-top:8px;color:#9ca3af;font-size:12px'>{escape(ui['event'])}</div></div>"""))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for raw in process.stdout:
        line = raw.strip()
        if not line:
            continue
        data = fields(line)
        if line.startswith('TRAIN '):
            ui.update(epoch=data['epoch'], shard=data['shard'], step=int(data['step']),
                      loss=data['loss'], mae=data.get('mae', '--'), lr=data.get('lr', '--'),
                      psnr=finite_metric(data.get('val_psnr')), best=finite_metric(data.get('best_psnr')),
                      data_s=data.get('data_s', '--'), compute_s=data.get('compute_s', '--'),
                      wall_s=data.get('wall_s', '--'), samples_s=data.get('samples_s', '--'))
            ui['epoch_batch'], ui['epoch_batches'] = map(int, data['epoch_batch'].split('/'))
            recent_wall.append(float(data['wall_s']))
            ui['eta'] = duration((ui['epoch_batches'] - ui['epoch_batch']) * sum(recent_wall) / len(recent_wall))
        elif line.startswith('STAGING '):
            ui['stage_percent'] = float(data['percent'])
            ui['stage'] = (f"Prefetch {data['name']}: {data['copied_gib']}/{data['total_gib']} GiB · "
                           f"{data['mib_s']} MiB/s · ETA {data['eta_s']} s")
        elif line.startswith('STAGED '):
            ui['stage_percent'] = 100.0
            ui['stage'] = line
        elif line.startswith('CHECKPOINT '):
            ui['checkpoint'] = int(data['step'])
            ui['event'] = f"Checkpoint saved to Drive at step {ui['checkpoint']:,}"
        elif line.startswith('INPUT_PIPELINE '):
            ui['decoder'] = data.get('decoder', 'auto')
            ui['event'] = line
        elif line.startswith('SHARD_COMPLETE '):
            ui['psnr'] = finite_metric(data.get('validation_psnr'))
            ui['best'] = finite_metric(data.get('best_psnr'))
            ui['event'] = line
        elif line.startswith(('PREFETCH_', 'SHARD_', 'RESUMED')):
            ui['event'] = line
        else:
            print(line, flush=True)
        render()
    returncode = process.wait()
    if returncode:
        render('#d32f2f')
        raise subprocess.CalledProcessError(returncode, cmd)
    ui['event'] = 'Training finished successfully.'
    ui['train_percent'] = 100.0
    render('#2e7d32')
